# Corporacion Favorita - Forecasting Model - 

## Holt-Winters Model Pipeline

Made by 4B Consultancy (Janne Heuvelmans, Georgi Duev, Alexander Engelage, Sebastiaan de Bruin) - 2024/2025

This notebook outlines the creation of a  pipeline for forecasting time-series sales data by using the Holt-Winters exponential smoothing model. 
Key functionalities include data preprocessing, model training, and batch-wise forecasting with a lag of 2 weeks and rolling forcasting window, followed by evaluation metrics.

The following steps are made within this notebook:  

>-0. Import Packages 

>-1. Load dataset and provide basic description
    >-1.1 Functions for Importing Data
    >-1.2 Import Raw Data

>-2. Data Splitting: Train, Test, Validation

>-3. Functions for Data Preprocessing 
    >-3.1 Impute Stockouts (missing data) based on product perishability
    >-3.2 Weekly Aggregation

>-4. Preprocessing Pipeline 
    -4.1 
    -4.2 

>-5. Holt-Winters Model Pipeline 
    >-5.1 Model Training for each unique store-item combination, extracting model parameters and configurations
    >-5.2 Rolling-window Forecast with Lag-2
    >-5.3 Evaluation Metrics using MAPE and Bias

## 0. Import Packages

In [1]:
import os
import sys
import time
from datetime import date, datetime, timedelta
import warnings

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_percentage_error

from statsmodels.tsa.holtwinters import ExponentialSmoothing
import statsmodels.api as sm

from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")

## 1. Load final dataset

### 1.1. Functions - Import raw data from local PATH
Create import data function and give basic information function within the importing function.

Return basic information on each dataframe:  
- a) Information on the number of observation and features.  
- b) Information on the size of the dataframe. 

In [6]:
def f_get_data_and_info(import_path, file_name):

    print(f"\nReading file {file_name}\n")

    # Load data.
    df = pd.read_parquet(import_path + file_name + ".parquet")

    # Getting the basic information of the dataframe (number of observations and features, and size)
    print(
        f"The '{file_name}' dataframe contains: {df.shape[0]:,}".replace(",", ".")
        + f" observations and {df.shape[1]} features."
    )
    print(
        f"Prepared and transformed dataframe has optimized size of {round(sys.getsizeof(df)/1024/1024/1024, 2)} GB."
    )

    return df

### 1.2. Importing raw data
Importing parquet files with importing function (giving basic information)

In [7]:
import_path = "C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE/"

# Importing final df
df_final = f_get_data_and_info(import_path, file_name="Prepped_data_20241211")

# Importing final train, test and validation df's
df_train = f_get_data_and_info(import_path, file_name="train_df_20241211")
df_test = f_get_data_and_info(import_path, file_name="test_df_20241211")
df_val = f_get_data_and_info(import_path, file_name="val_df_20241211")


Reading file Prepped_data_20241211

The 'Prepped_data_20241211' dataframe contains: 25.686.262 observations and 19 features.
Prepared and transformed dataframe has optimized size of 0.84 GB.


## 2.0 Train Test Val Split

train_test_val_split without creating X (features) and y (target)

In [8]:
def train_test_val_split(df, window_length=26):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "week_number_cum"])

    # Get the maximum week in the dataset
    max_week = df["week_number_cum"].max()

    # Calculate start and end weeks for test and validation  sets
    val_week_end = max_week - 1

    val_week_start = max_week - window_length

    test_week_start = max_week - 2 * window_length

    test_week_end = val_week_start - 1

    train_week_start = max_week - 6 * window_length

    train_week_end = test_week_start - 1

    # Train data: All data before the start of the validation period
    train = df[
        (df["week_number_cum"] >= train_week_start)
        & (df["week_number_cum"] <= train_week_end)
    ]

    # Val data: From val_week_start to val_week_end
    test = df[
        (df["week_number_cum"] >= test_week_start)
        & (df["week_number_cum"] <= test_week_end)
    ]

    # Test data: From test_week_start to max_week
    val = df[
        (df["week_number_cum"] >= val_week_start)
        & (df["week_number_cum"] <= val_week_end)
    ]

    # Function to print split information
    def print_split_info(split_name, split):
        print(f"\n{split_name} set: shape: {split.shape}")
        print(f"{split_name} Min Week: {split['week_number_cum'].min()}")
        print(f"{split_name} Min Date: {split['date'].min()}")
        print(f"{split_name} Max Week: {split['week_number_cum'].max()}")
        print(f"{split_name} Max Date: {split['date'].max()}")
        print(f"{split_name} number of weeks: {split['week_number_cum'].nunique()}")
        print(f"Number of stores: {split['store_nbr'].nunique()}")
        print(f"Number of items: {split['item_nbr'].nunique()}")
        print(f"Size of {round(sys.getsizeof(split)/1024/1024/1024, 2)} GB.")

    # Print information about the splits
    print_split_info("Train", train)
    print_split_info("Test", test)
    print_split_info("Validation", val)

    return train, test, val

## 3.0 Functions - Impute stockouts and Aggregate dataset to weekly level


#### 3.1. Impute stockouts

Stockout on store level

•      Perishable good: when there are missing values for two consecutive days for a given item per individual store 

•      Nonperishable goods: when there are missing values for 7 consecutive days for a given item and per individual store

•      Action: Impute with Rolling Mean with defeault window of 7 days 

------------------------------------

In [9]:
def impute_stockouts_polars(df_pandas, window_size=7):
    # Convert the input Pandas DataFrame to a Polars DataFrame
    df = pl.from_pandas(df_pandas)

    # Sort the DataFrame by store number, item number, and date for proper ordering
    df = df.sort(["store_nbr", "item_nbr", "date"])

    # Create a boolean column to indicate where 'unit_sales' is missing
    df = df.with_columns((pl.col("unit_sales").is_null()).alias("is_missing"))

    # Assign a group identifier to each segment of missing or non-missing values
    df = df.with_columns(
        (
            pl.col("is_missing").cast(pl.Int16)
            != pl.col("is_missing").cast(pl.Int16).shift(1)
        )
        .cast(pl.Int16)
        .cum_sum()
        .alias("missing_group")
        .cast(pl.Int32)
    )

    # Calculate cumulative count of missing values within each segment of missing data
    df = df.with_columns(
        pl.when(pl.col("is_missing"))
        .then(
            pl.col("is_missing")
            .cast(pl.Int16)
            .cum_sum()
            .over(["store_nbr", "item_nbr", "missing_group"])
        )
        .otherwise(0)
        .alias("missing_count")
    )

    # Identify groups to find maximum of the same missing_count group
    df = df.with_columns(
        pl.col("missing_count")
        .max()
        .over(["missing_group"])
        .alias("group_max_missing_count")
        .cast(pl.Int16)
    )

    # Function for rolling mean imputation
    def rolling_mean_imputation(df, window_size=7):
        # Add rolling mean column for grouped data
        df = df.with_columns(
            pl.col("unit_sales")
            .rolling_mean(window_size=window_size, min_periods=1)
            .over(["store_nbr", "item_nbr"])
            .shift(1)  # Shift to exclude current row
            .alias("unit_sales_rolling_mean")
        )

        # Replace nulls in the target column with the calculated rolling mean
        df = df.with_columns(
            pl.when(pl.col("unit_sales").is_null())
            .then(pl.col("unit_sales_rolling_mean"))
            .otherwise(pl.col("unit_sales"))
            .alias("unit_sales")
        )

        # Drop the temporary rolling mean column
        df = df.drop("unit_sales_rolling_mean")

        return df

    # Apply rolling mean imputation based on perishable status
    df = df.with_columns(
        [
            pl.when(pl.col("perishable") == 1)  # If the item is perishable
            .then(
                pl.when(pl.col("group_max_missing_count") == 1)  # 1 missing value
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") > 2
                )  # More than 2 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") == 2
                )  # Exactly 2 missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for 2 missing days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .when(pl.col("perishable") == 0)  # If the item is not perishable
            .then(
                pl.when(
                    pl.col("group_max_missing_count") > 7
                )  # More than 7 missing values
                .then(0)  # Impute with 0
                .when(
                    pl.col("group_max_missing_count") <= 7
                )  # 7 or fewer missing values
                .then(
                    rolling_mean_imputation(df, window_size=7)["unit_sales"]
                )  # Impute with rolling mean for missing 7 or fewer days
                .otherwise(pl.col("unit_sales"))  # Keep original value
            )
            .otherwise(pl.col("unit_sales"))  # For other cases, keep the original value
            .alias("unit_sales")
        ]
    )

    df = df.drop(
        "is_missing", "missing_group", "missing_count", "group_max_missing_count"
    )

    # Convert Polars df back to Pandas df
    df = df.to_pandas()

    return df

### 3.2. Aggregate dataset to weekly level

- Group the DataFrame by store number, item number, year, and week_cum_number, then aggregate the columns
--> "unit_sales","onpromotion", "holiday_local_count","holiday_regional_count","holiday_national_count",


In [10]:
def aggregate_week(df):

    # Sort the DataFrame by store number, item number, and date for ordering
    df = df.sort_values(["store_nbr", "item_nbr", "year", "week_nbr"])

    # Group by the specified columns and aggregate
    df = (
        df.groupby(
            [
                "store_nbr",
                "item_nbr",
                "year",
                "week_number_cum",  # Aggregating by week_number_cum
            ]
        )
        .agg(
            {
                "unit_sales": "sum",
                "onpromotion": "sum",
                "holiday_local_count": "sum",
                "holiday_regional_count": "sum",
                "holiday_national_count": "sum",
                "date": "first",  # Keep the first day of week, needed to run Timeseries models from SKtime
                "store_type": "first",  # Keep the first occurrence of store_type
                "store_cluster": "first",  # Keep the first occurrence of store_cluster
                "item_family": "first",  # Keep the first occurrence of item_family
                "item_class": "first",  # Keep the first occurrence of item_class
                "perishable": "first",  # Keep the first occurrence of perishable
                "store_status": "last",  # Keep the last occurrence of store_status
                "item_status": "last",  # Keep the last occurrence of item_status
            }
        )
        .reset_index()
    )

    return df

## 4. Pipeline and preprocessing

Splitting and preprocessing with imputation and aggregating to weekly data

In [11]:
features = [
    "store_nbr",
    "item_nbr",
    "date",
    "onpromotion",
    "store_type",
    "store_cluster",
    "item_family",
    "item_class",
    "perishable",
    "year",
    "week_number_cum",
]

target_variable = ["unit_sales"]

In [12]:
def impute_agg_preprocessing(df, window_size=7):

    df = impute_stockouts_polars(df, window_size)

    df = aggregate_week(df)

    return df

In [13]:
def preprocess_split_filter(df, features, target_variable):

    # Splitting in train, test, validation split
    print(f"\nStep 1: Splitting in train, test, validation split")
    train_df, test_df, val_df = train_test_val_split(df)

    # Preprocessing with imputation and aggregating to weekly data
    print(f"\nStep 2: Preprocessing with imputation and aggregating to weekly data")
    train_df = impute_agg_preprocessing(train_df)
    test_df = impute_agg_preprocessing(test_df)
    val_df = impute_agg_preprocessing(val_df)

    # Filter spilts on needed feature and target variables
    print(f"\nStep 3: Filter spilts on needed feature and target variables")
    train_df = train_df[features + target_variable]
    test_df = test_df[features + target_variable]
    val_df = val_df[features + target_variable]

    # Ensure df's are sorted by store, item, and date for alignment
    print(f"\nStep 4: Ensure dfs are sorted by store, item, and date for alignment")
    train_df = train_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    test_df = test_df.sort_values(by=["store_nbr", "item_nbr", "date"])
    val_df = val_df.sort_values(by=["store_nbr", "item_nbr", "date"])

    return train_df, test_df, val_df

In [ ]:
# df_final = df_final[
# (df_final["store_nbr"] == 48)
# ]  # & (df_final["item_nbr"] == 103520)]

In [15]:
train_df, test_df, val_df = preprocess_split_filter(df_final, features, target_variable)


Step 1: Splitting in train, test, validation split

Train set: shape: (11084528, 19)
Train Min Week: 86
Train Min Date: 2014-08-18 00:00:00
Train Max Week: 189
Train Max Date: 2016-08-14 00:00:00
Train number of weeks: 104
Number of stores: 39
Number of items: 550
Size of 0.44 GB.

Test set: shape: (2771132, 19)
Test Min Week: 190
Test Min Date: 2016-08-15 00:00:00
Test Max Week: 215
Test Max Date: 2017-02-12 00:00:00
Test number of weeks: 26
Number of stores: 39
Number of items: 550
Size of 0.11 GB.

Validation set: shape: (2771132, 19)
Validation Min Week: 216
Validation Min Date: 2017-02-13 00:00:00
Validation Max Week: 241
Validation Max Date: 2017-08-13 00:00:00
Validation number of weeks: 26
Number of stores: 39
Number of items: 550
Size of 0.11 GB.

Step 2: Preprocessing with imputation and aggregating to weekly data

Step 3: Filter spilts on needed feature and target variables

Step 4: Ensure dfs are sorted by store, item, and date for alignment


## 5. Model Pipeline

Holt Winters with unique_id

### 5.1 Model Fit seperated from Predicting/Forecasting function

- This function trains Holt-Winters models for each unique store-item combination in the dataset and stores their fitted models and parameters. Constructs a unique identifier (unique_id) for each store-item combination using store_nbr and item_nbr.

Inputs:

- Takes training data (train_df)
- Accepts model parameter configuration options: seasonal_periods, trend, damped_trend, seasonal, and smoothing parameters (alpha, beta, gamma, theta).
- Possible option to use package optimzing of model parameters with optimized and use_brute for brute grid search.


Outputs:

- params_df: Contains extracted parameters (alpha, beta, gamma, theta) for each fitted model.
- fitted_models: Stores the fitted Holt-Winters models for each unique_id containing model parameter configurations

In [19]:
def holt_winters_model_fit(
    train_df,
    seasonal_periods=None,
    trend=None,
    damped_trend=None,
    seasonal=None,
    alpha=None,
    beta=None,
    gamma=None,
    theta=None,
    optimized=False,
    use_brute=False,
):

    model_params = []
    fitted_models = {}

    # Create and store the current unique identifier for tracking
    train_df["unique_id"] = (
        train_df["store_nbr"].astype(str) + "_" + train_df["item_nbr"].astype(str)
    )

    # Extract all unique identifiers from the unique_id column
    unique_ids = train_df["unique_id"].unique()

    # Loop through each unique id
    for unique_id in unique_ids:

        # Extract store and item numbers
        store, item = unique_id.split("_")

        # Filter the data for the specific unique_id
        train_data = train_df[train_df["unique_id"] == unique_id].copy()

        # Convert date to datetime and set as index
        train_data["date"] = pd.to_datetime(train_data["date"])
        train_data = train_data.set_index("date")
        train_data = train_data.asfreq(
            "W-MON"
        )  # Setting frequency to weekly, starting week at Monday

        # Extract the unit_sales series
        train_series = train_data["unit_sales"].copy()

        # Fit Holt-Winters Model
        try:
            model = ExponentialSmoothing(
                train_series,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods,
            )

            fitted_model = model.fit(
                smoothing_level=alpha,
                smoothing_trend=beta,
                smoothing_seasonal=gamma,
                damping_trend=theta,
                optimized=optimized,
                use_brute=use_brute,
            )

            # Store the fitted model for each unique_id
            fitted_models[unique_id] = fitted_model

            # Extract model parameters
            params = {
                "unique_id": unique_id,
                "store_nbr": store,
                "item_nbr": item,
                "damping": fitted_model.params.get("damping_slope", None),
                "alpha": fitted_model.model.params.get("smoothing_level", None),
                "beta": fitted_model.model.params.get("smoothing_trend", None),
                "gamma": fitted_model.model.params.get("smoothing_seasonal", None),
                "theta": fitted_model.model.params.get("damping_trend", None),
            }
            model_params.append(params)

        except Exception as e:
            print(f"Model failed for unique_id {unique_id}: {e}")

    # Convert model parameters to df
    params_df = pd.DataFrame(model_params)
    params_df["store_nbr"] = params_df["store_nbr"].astype("int8")
    params_df["item_nbr"] = params_df["item_nbr"].astype("int32")

    print(
        "Model Fit processing complete. Trained/Fitted models per unique store-item id and params_df returned"
    )

    return params_df, fitted_models

In [17]:
# # Run model with Optimized=True,  use_brute=True
# train_params_df, fitted_models = holt_winters_model_fit(
#     train_df,
#     seasonal_periods=52,
#     trend="add",
#     damped_trend=True,
#     seasonal="add",
#     optimized=True,
#     use_brute=True
# )  # --> 56min to run

In [20]:
train_params_df, fitted_models = holt_winters_model_fit(
    train_df,
    seasonal_periods=52,
    trend="add",
    damped_trend=True,
    seasonal="add",
    alpha=0.1,
    beta=0.1,
    gamma=0.5,
    theta=0.5,
)

Model Fit processing complete. Trained/Fitted models per unique store-item id and params_df returned


In [71]:
train_params_df.head(5)

,unique_id,store_nbr,item_nbr,damping,alpha,beta,gamma,theta
0,48_103520,48,103520,None,0.1,0.1,0.5,0.5
1,48_105693,48,105693,None,0.1,0.1,0.5,0.5
2,48_106716,48,106716,None,0.1,0.1,0.5,0.5
3,48_108079,48,108079,None,0.1,0.1,0.5,0.5
4,48_108797,48,108797,None,0.1,0.1,0.5,0.5


#### 5.2.3. Forecast rolling window with Lag=2 loop in Batches

This function forecasts time-series data for multiple store-item combinations using pre-trained Holt-Winters models, constructs a unique_id for each store-item combination, organizes the predictions with Lag of 2 weeks with a rolling time window.

Divides the unique id's into batches of size 20 (or smaller for fewer items) for efficient processing to forecast the rolling predictions.


Input Parameters:

- Uses training (train_df), testing (test_df), and optionally validation data (val_df).
- Pre-trained Holt-Winters models (fitted_models) containing pre-definied parameters
- An output directory for saving results

Rolling Forecast (For each unique_id):

- Extracts the pre-trained model and the associated training and forecasting data
- Applies a rolling approach to update the training data with observed values (addition of week) as predictions are made iteratively
- Forecasts t+2 weeks ahead using the fitted Holt-Winters model while replacing negative forecasts with zero

Output:

- Merges true observed values with predictions for each store-item combination.
- Stores these combined results batch-wise in parquet files for intermediate storage.
- Merges all batch files into a single file (all_predictions.parquet) and saves it in the specified output directory.

In [21]:
def holt_winters_forecast(
    train_df=train_df,
    test_df=test_df,
    val_df=None,
    fitted_models=fitted_models,
    output_dir=not None,
):

    # Create and store the current unique identifier
    train_df["unique_id"] = (
        train_df["store_nbr"].astype(str) + "_" + train_df["item_nbr"].astype(str)
    )
    test_df["unique_id"] = (
        test_df["store_nbr"].astype(str) + "_" + test_df["item_nbr"].astype(str)
    )

    if val_df is not None:
        val_df["unique_id"] = (
            val_df["store_nbr"].astype(str) + "_" + val_df["item_nbr"].astype(str)
        )

    # Extract all unique identifiers from the unique_id column
    unique_ids = train_df["unique_id"].unique()

    # Calculate batch size, ensure batch_size cam be at least 1 and devide into batches of max 20
    batch_size = max(len(unique_ids) // 20, 1)
    batches = [
        unique_ids[i : i + batch_size] for i in range(0, len(unique_ids), batch_size)
    ]

    # Process each batch
    for batch_index, batch in enumerate(batches):
        print(f"\nProcessing batch {batch_index + 1} of {len(batches)}")

        # Initialize batch-specific predictions storage
        batch_predictions = []

        # Progress bar for the batch
        with tqdm(
            total=len(batch), desc=f"Batch {batch_index + 1}", unit="group"
        ) as pbar:

            # Loop through each unique id in the current batch
            for unique_id in batch:

                # Filter the train_df and forecast_df for the current unique_id
                train_data = train_df[train_df["unique_id"] == unique_id]

                forecast_data = test_df[test_df["unique_id"] == unique_id]

                # Check if val_df is provided, then test data needs to combined with test data
                if (
                    val_df is not None
                ):  # So forecast_data contains val_df (if val_df is provided) or test_df otherwise

                    # Combine train_data with test_data
                    train_data = pd.concat(
                        [train_data, forecast_data], axis=0, ignore_index=True
                    )

                    # Filter the val_df for the current unique_id and overwrite/assign as forecast_data
                    forecast_data = val_df[val_df["unique_id"] == unique_id]

                # Convert date to datetime and set as index
                train_data["date"] = pd.to_datetime(train_data["date"])
                train_data = train_data.set_index("date")
                train_data = train_data.asfreq("W-MON")

                forecast_data["date"] = pd.to_datetime(forecast_data["date"])
                forecast_data = forecast_data.set_index("date")
                forecast_data = forecast_data.asfreq("W-MON")

                # DEBUG / CHECK
                # print(
                #     f"\nTrain Data - Week Number Cum: Min = {train_data['week_number_cum'].min()}, Max = {train_data['week_number_cum'].max()}"
                # )
                # print(
                #     f"Forecast Data - Week Number Cum: Min = {forecast_data['week_number_cum'].min()}, Max = {forecast_data['week_number_cum'].max()}"
                # )

                # Rolling train data
                rolling_train = train_data.copy()

                # Extract indexed dates for test sets
                forecast_dates = forecast_data.index
                forecast_length = len(forecast_data)

                # Initialize lists to store predictions
                t_plus_2_pred = []
                pred_plus_2_dates = []

                # Extract store and item numbers from unique_id
                store, item = unique_id.split("_")
                store = np.int8(store)
                item = np.int32(item)

                # Extract the pre-trained model for the specific unique_id with pre-defined parameters
                fitted_model = fitted_models[unique_id]

                # Iteratively forecast t+2 with pre-trained model with pre-defined parameters
                for i in range(forecast_length):
                    try:

                        # Initialize a new model using pre-defined parameters from fitte_models and the rolling training data
                        model = ExponentialSmoothing(
                            rolling_train["unit_sales"],
                            trend=fitted_model.model.trend,
                            damped_trend=fitted_model.model.damped_trend,
                            seasonal=fitted_model.model.seasonal,
                            seasonal_periods=fitted_model.model.seasonal_periods,
                        )

                        # Re-Fit the model using parameters from the pre-trained model
                        refitted_model = model.fit(
                            smoothing_level=fitted_model.model.params.get(
                                "smoothing_level"
                            ),
                            smoothing_slope=fitted_model.model.params.get(
                                "smoothing_trend"
                            ),
                            smoothing_seasonal=fitted_model.model.params.get(
                                "smoothing_seasonal"
                            ),
                            damping_trend=fitted_model.model.params.get(
                                "damping_trend"
                            ),
                            optimized=False,  # Ensure the same pre-defined parameters are used
                        )

                        # Generate t+2 forecast
                        forecast = refitted_model.forecast(2)  # Forecast t+2
                        forecast_plus_2 = forecast.iloc[
                            -1
                        ]  # Only save t+2 forecast value
                        forecast_plus_2 = np.maximum(  # Replace negative values with 0
                            forecast_plus_2, 0
                        )

                        # Append the forecast to the list
                        t_plus_2_pred.append(forecast)

                        # Append the correct t+2 date
                        date_plus_2 = forecast_dates[i] + pd.Timedelta(weeks=1)
                        pred_plus_2_dates.append((date_plus_2, forecast_plus_2))

                        # Add the observed true test (t) to the rolling training data
                        rolling_train = pd.concat(
                            [rolling_train, forecast_data.iloc[[i]]]
                        )

                    except Exception as e:
                        print(
                            f"\nModel failed for unique_id {unique_id} for week {i + 1}: {e}"
                        )

                # Combine the test dates with the predictions and actual sales
                # Create a df for the true values
                true_df = pd.DataFrame(
                    {
                        "store_nbr": store,
                        "item_nbr": item,
                        "date": forecast_dates,
                        "y_true": forecast_data["unit_sales"].values,
                    }
                )

                # Create a df for the predictions
                pred_df = pd.DataFrame(
                    {
                        "store_nbr": store,
                        "item_nbr": item,
                        "date": [pred[0] for pred in pred_plus_2_dates],
                        "y_pred": [pred[1] for pred in pred_plus_2_dates],
                    }
                )
                # Merge the true and predicted values
                predictions_df = true_df.merge(
                    pred_df,
                    on=["date", "store_nbr", "item_nbr"],
                    # on=["date"],
                    how="inner",
                )

                # Append the predictions for this unique_id item-store combination to the batch list
                batch_predictions.append(predictions_df)

                # Update the progress bar
                pbar.update(1)

        # Export batch predictions
        batch_predictions_df = pd.concat(batch_predictions, ignore_index=True)

        # Save each batch to a parquet file
        output_file = os.path.join(
            output_dir, f"predictions_batch_{batch_index + 1}.parquet"
        )
        batch_predictions_df.to_parquet(output_file, index=False)
        print(f"Saved predictions for batch {batch_index + 1}")

    # Merge all batch files after processing
    predictions_file_template = os.path.join(output_dir, "predictions_batch_{}.parquet")
    all_predictions = pd.concat(
        [
            pd.read_parquet(predictions_file_template.format(i + 1))
            for i in range(len(batches))
        ],
        ignore_index=True,
    )

    # Save the final combined results
    all_predictions.to_parquet(
        os.path.join(output_dir, "all_predictions.parquet"),
        index=False,
        engine="fastparquet",
    )

    print("\nBatch processing complete. Combined results saved saved to {output_dir}.")

    return all_predictions

---------------------------------------------------------

- Train Data - Week Number Cum: Min = 86, Max = 189
- Forecast Data - Week Number Cum: Min = 190, Max = 215

In [ ]:
# Predict test
all_predictions = holt_winters_forecast(
    train_df=train_df,
    test_df=test_df,
    val_df=None,
    fitted_models=fitted_models,
    output_dir="C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE",
)

- Train Data - Week Number Cum: Min = 86, Max = 215
- Forecast Data - Week Number Cum: Min = 216, Max = 241

In [22]:
# Predict val
all_predictions = holt_winters_forecast(
    train_df=train_df,
    test_df=test_df,
    val_df=val_df,
    fitted_models=fitted_models,
    output_dir="C:/Users/alexander/Documents/0. Data Science and AI for Experts/FILE",
)


Processing batch 1 of 21


Batch 1: 100%|██████████| 761/761 [05:57<00:00,  2.13group/s]


Saved predictions for batch 1

Processing batch 2 of 21


Batch 2: 100%|██████████| 761/761 [05:01<00:00,  2.52group/s]


Saved predictions for batch 2

Processing batch 3 of 21


...

Saved predictions for batch 19

Processing batch 20 of 21


Batch 20: 100%|██████████| 761/761 [04:59<00:00,  2.54group/s]


Saved predictions for batch 20

Processing batch 21 of 21


Batch 21: 100%|██████████| 6/6 [00:02<00:00,  2.40group/s]


Saved predictions for batch 21

Batch processing complete. Combined results saved saved to {output_dir}.


### 5.4. Evaulation Metrics and Evaluate Model functions

In [23]:
def calculate_rmse(df, actual_col, predicted_col):
    mse = np.mean((df[actual_col] - df[predicted_col]) ** 2)  # Mean Squared Error
    rmse = np.sqrt(mse)  # Root Mean Squared Error
    return print(f"Average RMSE across all store-item combinations: {rmse}")

In [24]:
calculate_rmse(all_predictions, actual_col="y_true", predicted_col="y_pred")

Average RMSE across all store-item combinations: 102.17369846801853


In [ ]:
def calculate_rmse_with_bias(df, actual_col, predicted_col):

    # Calculate differences
    differences = df[predicted_col] - df[actual_col]

    # Calculate RMSE
    mse = np.mean(differences**2)  # Mean Squared Error
    rmse = np.sqrt(mse)  # Root Mean Squared Error

    # Calculate positive and negative biases
    positive_bias = (
        differences[differences > 0].mean() if (differences > 0).any() else 0
    )
    negative_bias = (
        differences[differences < 0].mean() if (differences < 0).any() else 0
    )

    # Print results
    print(f"Average RMSE: {rmse}")
    print(f"Mean Positive Bias: {positive_bias}")
    print(f"Mean Negative Bias: {negative_bias}")

    # Return results in a dictionary
    return {
        "RMSE": rmse,
        "Mean Positive Bias": positive_bias,
        "Mean Negative Bias": negative_bias,
    }

-------------------------------------------